<a href="https://colab.research.google.com/github/Mayar215999/data-science-project/blob/main/etl_project_gdp_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

✅ Step 1: Open Google Colab

✅ Step 2: Install Required Libraries (if needed)

In [1]:
!pip install beautifulsoup4 pandas numpy requests


✅ Step 3: Import Libraries

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import logging


✅ Step 4: Set Up Logging

In [3]:
# Setup logging
logging.basicConfig(filename='etl_project_log.txt',
                    level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')


✅ Step 5: Define the ETL Functions

In [5]:
def extract(url):
    logging.info("Starting data extraction.")
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    tables = pd.read_html(str(soup))
    for i, table in enumerate(tables):
        print(f"Table {i} Columns:\n{table.columns}\n")
    return tables


In [6]:
def transform(df):
    logging.info("Starting data transformation.")
    df = df[['Country/Territory', ('IMF[1][12]', 'Forecast')]]
    df.columns = ['Country', 'GDP_USD_million']
    df['GDP_USD_billion'] = np.round(df['GDP_USD_million'] / 1000, 2)
    df = df[['Country', 'GDP_USD_billion']]
    return df


✅ Step 6: Load Data to CSV and SQLite

In [7]:
def load(df):
    logging.info("Loading data to CSV and database.")
    df.to_csv("Countries_by_GDP.csv", index=False)

    conn = sqlite3.connect("World_Economies.db")
    df.to_sql("Countries_by_GDP", conn, if_exists="replace", index=False)

    query = "SELECT * FROM Countries_by_GDP WHERE GDP_USD_billion > 100"
    result_df = pd.read_sql(query, conn)
    conn.close()

    print("Countries with GDP > 100 billion USD:\n")
    print(result_df)


✅ Step 7: Run the ETL Pipeline

In [8]:
try:
    url = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"
    logging.info("ETL Job Started")

    tables = extract(url)
    df_raw = tables[2]  # adjust if needed after checking output
    df_transformed = transform(df_raw)
    load(df_transformed)

    logging.info("ETL Job Completed Successfully.")
except Exception as e:
    logging.error(f"ETL job failed: {e}")
    print("ETL job failed. Check log for details.")


<ipython-input-5-00e45c9f8b6f>:5: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
ERROR:root:ETL job failed: "[('IMF[1][12]', 'Forecast')] not in index"


Table 0 Columns:
Index([0], dtype='int64')

Table 1 Columns:
Index([0, 1, 2], dtype='int64')

Table 2 Columns:
MultiIndex([( 'Country/Territory', 'Country/Territory'),
            (         'UN region',         'UN region'),
            (        'IMF[1][13]',          'Estimate'),
            (        'IMF[1][13]',              'Year'),
            (    'World Bank[14]',          'Estimate'),
            (    'World Bank[14]',              'Year'),
            ('United Nations[15]',          'Estimate'),
            ('United Nations[15]',              'Year')],
           )

Table 3 Columns:
Index(['vteLists of countries by financial rankings', 'vteLists of countries by financial rankings.1'], dtype='object')

Table 4 Columns:
Index(['vteLists of countries by GDP rankings', 'vteLists of countries by GDP rankings.1'], dtype='object')

Table 5 Columns:
Index(['vteEconomic classification of countries', 'vteEconomic classification of countries.1'], dtype='object')

Table 6 Columns:
Index([

Perfect! Based on the structure you shared, here is a complete and organized version of etl_project_gdp.py that you can run directly in Google Colab or save as a Python script:

✅ Full ETL Script (Structured Format)

The code should be created in an organized manner such that you can perform each task with a dedicated function. For reference, you can copy paste the structure as shown below to etl_project_gdp.py

In [12]:
# Code for ETL operations on Country-GDP data

# Importing the required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import logging

# Logging setup
logging.basicConfig(filename='etl_project_log.txt',
                    level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

def log_progress(message):
    ''' This function logs the mentioned message at a given stage of the code execution to a log file. '''
    logging.info(message)

def extract(url, table_attribs):
    ''' Extract data from the website and return as a dataframe. '''
    log_progress("Starting data extraction.")
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    tables = pd.read_html(str(soup))

    # Loop through to find correct table by matching column names
    for table in tables:
        if all(attr in table.columns for attr in table_attribs):
            log_progress("Correct table found and extracted.")
            return table
    raise ValueError("Required table with specified attributes not found.")

def transform(df):
    ''' Convert GDP from millions to billions, round to 2 decimal places, and rename columns. '''
    log_progress("Starting data transformation.")
    df = df[['Country/Territory', ('IMF[1][12]', 'Forecast')]]
    df.columns = ['Country', 'GDP_USD_million']
    df['GDP_USD_billion'] = np.round(df['GDP_USD_million'] / 1000, 2)
    df = df[['Country', 'GDP_USD_billion']]
    log_progress("Data transformation completed.")
    return df

def load_to_csv(df, csv_path):
    ''' Save the dataframe as a CSV file. '''
    df.to_csv(csv_path, index=False)
    log_progress(f"Data loaded to CSV at {csv_path}.")

def load_to_db(df, sql_connection, table_name):
    ''' Save the dataframe as a table in the database. '''
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)
    log_progress(f"Data loaded to table '{table_name}' in database.")

def run_query(query_statement, sql_connection):
    ''' Run a query on the database and print the result. '''
    log_progress("Running query on database.")
    result = pd.read_sql(query_statement, sql_connection)
    print("Query Result:\n", result)
    log_progress("Query executed successfully.")

# --------------------------------------------
# Main ETL Process
# --------------------------------------------
try:
    log_progress("ETL job started.")

    # Define URL and parameters
    url = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"
    table_attribs = ['Country/Territory', ('IMF[1][12]', 'Forecast')]

    # Extract
    df_extracted = extract(url, table_attribs)

    # Transform
    df_transformed = transform(df_extracted)

    # Load to CSV
    csv_path = "Countries_by_GDP.csv"
    load_to_csv(df_transformed, csv_path)

    # Load to Database
    db_path = "World_Economies.db"
    conn = sqlite3.connect(db_path)
    load_to_db(df_transformed, conn, "Countries_by_GDP")

    # Query
    query = "SELECT * FROM Countries_by_GDP WHERE GDP_USD_billion > 100"
    run_query(query, conn)
    conn.close()

    log_progress("ETL job completed successfully.")

except Exception as e:
    log_progress(f"ETL job failed: {e}")
    print(f"ETL job failed. Check log for details: {e}")


ETL job failed. Check log for details: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7ae76a73f450>, 'Connection to web.archive.org timed out. (connect timeout=None)'))


✅ Preliminary Setup for etl_project_gdp.py

In [13]:
# Code for ETL operations on Country-GDP data

# -----------------------------------
# Importing the required libraries
# -----------------------------------
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import logging

# -----------------------------------
# Logging setup
# -----------------------------------
logging.basicConfig(filename='etl_project_log.txt',
                    level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

def log_progress(message):
    ''' This function logs the mentioned message at a given stage of the code execution to a log file. '''
    logging.info(message)

# -----------------------------------
# Initializing known values
# -----------------------------------
log_progress("Initializing known variables.")

# URL of the GDP data from Wikipedia (archived)
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'

# Columns to look for in the HTML table
table_attribs = ['Country/Territory', ('IMF[1][12]', 'Forecast')]

# CSV file path
csv_path = 'Countries_by_GDP.csv'

# SQLite database and table name
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'

log_progress("Known variables initialized.")


In [14]:
# Code for ETL operations on Country-GDP data

# -----------------------------------
# Importing the required libraries
# -----------------------------------
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import logging

# -----------------------------------
# Logging setup
# -----------------------------------
logging.basicConfig(filename='etl_project_log.txt',
                    level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

def log_progress(message):
    ''' This function logs the mentioned message at a given stage of the code execution to a log file. '''
    logging.info(message)

# -----------------------------------
# Initializing known values
# -----------------------------------
log_progress("Initializing known variables.")

# URL to scrape GDP data
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'

# Expected columns in the initial DataFrame (before transformation)
table_attribs = ["Country", "GDP_USD_millions"]

# Output paths
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'
csv_path = './Countries_by_GDP.csv'

log_progress("Known variables initialized.")


task 1 : Extracting

In [15]:
def extract(url, table_attribs):
    ''' This function extracts the required
    information from the website and saves it to a dataframe. The
    function returns the dataframe for further processing. '''

    # Get page content
    page = requests.get(url).text

    # Parse using BeautifulSoup
    data = BeautifulSoup(page, 'html.parser')

    # Initialize empty dataframe with column names
    df = pd.DataFrame(columns=table_attribs)

    # Get all table bodies
    tables = data.find_all('tbody')

    # Select the third table (index 2)
    rows = tables[2].find_all('tr')

    # Loop through each row to extract valid data
    for row in rows:
        col = row.find_all('td')
        if len(col) != 0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {
                    "Country": col[0].a.contents[0],
                    "GDP_USD_millions": col[2].contents[0]
                }
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df, df1], ignore_index=True)

    log_progress("Data extraction complete. Number of valid rows extracted: {}".format(len(df)))
    return df


Task 2: Transform information

In [16]:
def transform(df):
    ''' This function converts the GDP information from Currency
    format to float value, transforms the information of GDP from
    USD (Millions) to USD (Billions) rounding to 2 decimal places.
    The function returns the transformed dataframe.'''

    # Step 1: Convert 'GDP_USD_millions' column to list
    GDP_list = df["GDP_USD_millions"].tolist()

    # Step 2: Remove commas, convert to float
    GDP_list = [float("".join(x.split(','))) for x in GDP_list]

    # Step 3: Convert from millions to billions, round to 2 decimal places
    GDP_list = [np.round(x / 1000, 2) for x in GDP_list]

    # Step 4: Replace old column with transformed data
    df["GDP_USD_millions"] = GDP_list

    # Step 5: Rename the column
    df = df.rename(columns={"GDP_USD_millions": "GDP_USD_billions"})

    # Log transformation progress
    log_progress("Data transformation complete. Converted GDP to billions.")

    return df

 Task 3: Loading information:



In [18]:
def load_to_csv(df, csv_path):
    ''' This function saves the final dataframe as a `CSV` file
    in the provided path. Function returns nothing. '''

    df.to_csv(csv_path, index=False)
    log_progress(f"Data successfully written to CSV at {csv_path}")


In [19]:
def load_to_db(df, sql_connection, table_name):
    ''' This function saves the final dataframe as a database table
    with the provided name. Function returns nothing. '''

    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)
    log_progress(f"Data successfully written to database table: {table_name}")


Task 4: Querying the database table

In [20]:
def run_query(query_statement, sql_connection):
    ''' This function runs the stated query on the database table and
    prints the output on the terminal. Function returns nothing. '''

    print(query_statement)  # For visibility/debugging
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)


✅ log_progress() function

In [21]:
def log_progress(message):
    timestamp_format = '%Y-%h-%d-%H:%M:%S'  # Year-Monthname-Day-Hour-Minute-Second
    now = datetime.now()  # get current timestamp
    timestamp = now.strftime(timestamp_format)
    with open("./etl_project_log.txt", "a") as f:
        f.write(timestamp + ' : ' + message + '\n')


In [24]:
def extract(url, table_attribs):
    ''' This function extracts the required
    information from the website and saves it to a dataframe. The
    function returns the dataframe for further processing. '''

    # Get page content
    # Added timeout parameter to requests.get()
    page = requests.get(url, timeout=30).text

    # Parse using BeautifulSoup
    data = BeautifulSoup(page, 'html.parser')

    # Initialize empty dataframe with column names
    df = pd.DataFrame(columns=table_attribs)

    # Get all table bodies
    tables = data.find_all('tbody')

    # Select the third table (index 2)
    rows = tables[2].find_all('tr')

    # Loop through each row to extract valid data
    for row in rows:
        col = row.find_all('td')
        if len(col) != 0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {
                    "Country": col[0].a.contents[0],
                    "GDP_USD_millions": col[2].contents[0]
                }
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df, df1], ignore_index=True)

    log_progress("Data extraction complete. Number of valid rows extracted: {}".format(len(df)))
    return df

Terminal Output:
Successful Execution Logs: After executing the command:

In [25]:
!python3.11 etl_project_gdp.py


python3.11: can't open file '/content/etl_project_gdp.py': [Errno 2] No such file or directory


In [26]:
# This code does nothing, as the provided text is not Python code.
# It appears to be log messages.  To make this executable Python code,
# you would need to define a function that generates the messages.

def generate_log_messages():
    messages = [
        "<timestamp> : Preliminaries complete. Initiating ETL process",
        "<timestamp> : Data extraction complete. Initiating Transformation process",
        "<timestamp> : Data transformation complete. Initiating loading process",
        "<timestamp> : Data saved to CSV file",
        "<timestamp> : SQL Connection initiated.",
        "<timestamp> : Data loaded to Database as table. Running the query",
        "<timestamp> : Process Complete."
    ]
    return messages

if __name__ == '__main__':
    # This code does nothing, as the provided text is not Python code.
    # It appears to be log messages.  To make this executable Python code,
    # you would need to define a function that generates the messages.

    #The second definition was nested in the first if statement and was causing the error.
    #This function has been removed, as it was a duplicate.
    #def generate_log_messages():
    #    messages = [
    #        "<timestamp> : Preliminaries complete. Initiating ETL process",
    #        "<timestamp> : Data extraction complete. Initiating Transformation process",
    #        "<timestamp> : Data transformation complete. Initiating loading process",
    #        "<timestamp> : Data saved to CSV file",
    #        "<timestamp> : SQL Connection initiated.",
    #        "<timestamp> : Data loaded to Database as table. Running the query",
    #        "<timestamp> : Process Complete."
    #    ]
    #    return messages

    #The second if __name__ == '__main__' was also nested in the first if statement and causing problems.
    #It and the for loop underneath it has been correctly indented.
    for message in generate_log_messages():
        print(message)

<timestamp> : Preliminaries complete. Initiating ETL process
<timestamp> : Data extraction complete. Initiating Transformation process
<timestamp> : Data transformation complete. Initiating loading process
<timestamp> : Data saved to CSV file
<timestamp> : SQL Connection initiated.
<timestamp> : Data loaded to Database as table. Running the query
<timestamp> : Process Complete.


In [24]:
!requests.exceptions.ConnectionError: HTTPSConnectionPool(host=’web.archive.org’, port=443): Max retries exceeded with url.


/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `requests.exceptions.ConnectionError: HTTPSConnectionPool(host=’web.archive.org’, port=443): Max retries exceeded with url.'


In [27]:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'


In [7]:
GDP_list = []
for x in df["GDP_USD_millions"]:
    try:
        clean_value = float("".join(x.split(',')))
        GDP_list.append(np.round(clean_value / 1000, 2))
    except Exception as e:
        GDP_list.append(np.nan)  # or handle however you prefer


In [28]:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'


In [29]:
def extract_from_html_file(html_file_path, table_attribs):
    with open(html_file_path, 'r', encoding='utf-8') as file:
        page = file.read()
    data = BeautifulSoup(page, 'html.parser')
    # [rest of your parsing code remains the same...]


In [32]:
def extract(url, table_attribs):
    ''' This function extracts the required
    information from the website and saves it to a dataframe. The
    function returns the dataframe for further processing. '''

    # Get page content
    # Added timeout parameter to requests.get()
    page = requests.get(url, timeout=30).text

    # Parse using BeautifulSoup
    data = BeautifulSoup(page, 'html.parser')

    # Initialize empty dataframe with column names
    df = pd.DataFrame(columns=table_attribs)

    # Get all table bodies
    tables = data.find_all('tbody')

    # Select the third table (index 2)
    rows = tables[2].find_all('tr')

    # Loop through each row to extract valid data
    for row in rows:
        col = row.find_all('td')
        if len(col) != 0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {
                    "Country": col[0].a.contents[0],
                    "GDP_USD_millions": col[2].contents[0]
                }
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df, df1], ignore_index=True)

    log_progress("Data extraction complete. Number of valid rows extracted: {}".format(len(df)))
    return df

In [34]:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'


In [35]:
from google.colab import files
uploaded = files.upload()


Saving List of countries by GDP (nominal) - Wikipedia.html to List of countries by GDP (nominal) - Wikipedia.html


In [37]:
from google.colab import files
uploaded = files.upload()

from bs4 import BeautifulSoup

# Access the uploaded file using its key in the 'uploaded' dictionary
# Assuming 'List_of_countries_by_GDP_(nominal).html' was the filename uploaded.
with open(list(uploaded.keys())[0], 'r', encoding='utf-8') as file:
    html = file.read()

soup = BeautifulSoup(html, 'html.parser')
# Continue parsing soup as usual

Saving List of countries by GDP (nominal) - Wikipedia.html to List of countries by GDP (nominal) - Wikipedia (1).html


In [38]:
import os
os.listdir()


['.config',
 'etl_project_log.txt',
 'List of countries by GDP (nominal) - Wikipedia (1).html',
 'drive',
 'List of countries by GDP (nominal) - Wikipedia.html',
 'sample_data']

In [41]:
with open('List of countries by GDP (nominal) - Wikipedia.html', 'r', encoding='utf-8') as file:
    html = file.read()


In [42]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, 'html.parser')
# Now you can extract tables or other elements as needed


In [43]:
from bs4 import BeautifulSoup
import pandas as pd

# Load the HTML file
with open('List of countries by GDP (nominal) - Wikipedia.html', 'r', encoding='utf-8') as file:
    html = file.read()

# Parse with BeautifulSoup
soup = BeautifulSoup(html, 'html.parser')


In [44]:
tables = soup.find_all('table', {'class': 'wikitable'})

print(f'Number of tables found: {len(tables)}')


Number of tables found: 1


In [45]:
# Preview the first table
print(tables[0].prettify()[:1000])  # Preview the first 1000 characters


<table border="1" class="wikitable sortable static-row-numbers plainrowheaders srn-white-background" style="text-align:right;">
 <caption>
  GDP (USD million) by country
 </caption>
 <tbody>
  <tr class="static-row-header" style="text-align:center;vertical-align:bottom;">
   <th rowspan="2">
    Country/Territory
   </th>
   <th rowspan="2">
    <a href="/web/20230902185326/https://en.wikipedia.org/wiki/United_Nations_geoscheme" title="United Nations geoscheme">
     UN region
    </a>
   </th>
   <th colspan="2">
    <a href="/web/20230902185326/https://en.wikipedia.org/wiki/International_Monetary_Fund" title="International Monetary Fund">
     IMF
    </a>
    <sup class="reference" id="cite_ref-GDP_IMF_2-2">
     <a href="#cite_note-GDP_IMF-2">
      [1]
     </a>
    </sup>
    <sup class="reference" id="cite_ref-15">
     <a href="#cite_note-15">
      [13]
     </a>
    </sup>
   </th>
   <th colspan="2">
    <a href="/web/20230902185326/https://en.wikipedia.org/wiki/World_Bank" 

In [46]:
df = pd.read_html(str(tables[0]))[0]  # Replace 0 with the correct index if needed


<ipython-input-46-4c6ae00a4304>:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tables[0]))[0]  # Replace 0 with the correct index if needed


In [47]:
df.columns = df.columns.droplevel(0) if isinstance(df.columns, pd.MultiIndex) else df.columns
df = df.dropna(how='all')  # Drop empty rows
df = df.rename(columns=lambda x: x.strip())  # Strip whitespace from column names


In [48]:
print(df.head())


  Country/Territory UN region   Estimate       Year   Estimate       Year  \
0             World         —  105568776       2023  100562011       2022   
1     United States  Americas   26854599       2023   25462700       2022   
2             China      Asia   19373586  [n 1]2023   17963171  [n 3]2022   
3             Japan      Asia    4409738       2023    4231141       2022   
4           Germany    Europe    4308854       2023    4072192       2022   

   Estimate       Year  
0  96698005       2021  
1  23315081       2021  
2  17734131  [n 1]2021  
3   4940878       2021  
4   4259935       2021  


In [49]:
df.head()


,Country/Territory,UN region,Estimate,Year,Estimate,Year,Estimate,Year
0,World,—,105568776,2023,100562011,2022,96698005,2021
1,United States,Americas,26854599,2023,25462700,2022,23315081,2021
2,China,Asia,19373586,[n 1]2023,17963171,[n 3]2022,17734131,[n 1]2021
3,Japan,Asia,4409738,2023,4231141,2022,4940878,2021
4,Germany,Europe,4308854,2023,4072192,2022,4259935,2021


In [51]:
# Remove rows with missing values in key columns
df = df.dropna(subset=[df.columns[1]])

# Rename columns for simplicity
df.columns = ['Country', 'GDP_USD_Millions'] + list(df.columns[2:])

# Remove commas, footnotes, and convert to numeric
df['GDP_USD_Millions'] = df['GDP_USD_Millions'].replace(r'\[.*\]', '', regex=True)

# Remove non-numeric characters and convert to float, handling errors
df['GDP_USD_Millions'] = pd.to_numeric(df['GDP_USD_Millions'].str.replace(',', '', regex=True).str.replace('—', '', regex=True), errors='coerce')

# Drop rows with NaN values in 'GDP_USD_Millions'
df = df.dropna(subset=['GDP_USD_Millions'])

df['GDP_USD_Millions'] = df['GDP_USD_Millions'].astype(float)

In [52]:
df.to_csv('gdp_by_country.csv', index=False)
print("Saved to gdp_by_country.csv")


Saved to gdp_by_country.csv


In [55]:
def log_progress(message):
    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second
    now = datetime.now() # get current timestamp
    timestamp = now.strftime(timestamp_format)
    with open("./etl_project_log.txt","a") as f:
        f.write(timestamp + ' : ' + message + '\n')

In [57]:
def transform(df):
    ''' This function converts the GDP information from Currency
    format to float value, transforms the information of GDP from
    USD (Millions) to USD (Billions) rounding to 2 decimal places.
    The function returns the transformed dataframe.'''

    # Create a new column called 'GDP_USD_billions' with initial values as NaN
    df['GDP_USD_billions'] = np.nan

    # Iterate over the DataFrame using .iterrows()
    for index, row in df.iterrows():
        try:
            # Convert, round, and assign to 'GDP_USD_billions'
            df.loc[index, 'GDP_USD_billions'] = round(float("".join(row['GDP_USD_Millions'].split(','))) / 1000, 2)  # Millions to billions
        except (ValueError, TypeError):
            # Handle any conversion errors
            log_progress(f"Error transforming GDP for country: {row['Country']}")  # Log the error
            # Optionally: df.loc[index, 'GDP_USD_billions'] = np.nan  # or keep as NaN

    # Log transformation progress
    log_progress("Data transformation complete. Converted GDP to billions.")

    # Select columns for the final DataFrame
    df_transformed = df[['Country', 'GDP_USD_billions']]

    return df_transformed

In [60]:
def transform(df):
    ''' This function converts the GDP information from Currency
    format to float value, transforms the information of GDP from
    USD (Millions) to USD (Billions) rounding to 2 decimal places.
    The function returns the transformed dataframe.'''

    # Create a new column called 'GDP_USD_billions' with initial values as NaN
    df['GDP_USD_billions'] = np.nan

    # Iterate over the DataFrame using .iterrows()
    for index, row in df.iterrows():
        try:
            # Convert, round, and assign to 'GDP_USD_billions'
            # Accessing 'GDP_USD_millions' instead of 'GDP_USD_Millions'
            df.loc[index, 'GDP_USD_billions'] = round(float("".join(row['GDP_USD_millions'].split(','))) / 1000, 2)  # Millions to billions
        except (ValueError, TypeError):
            # Handle any conversion errors
            log_progress(f"Error transforming GDP for country: {row['Country']}")  # Log the error
            # Optionally: df.loc[index, 'GDP_USD_billions'] = np.nan  # or keep as NaN

    # Log transformation progress
    log_progress("Data transformation complete. Converted GDP to billions.")

    # Select columns for the final DataFrame
    df_transformed = df[['Country', 'GDP_USD_billions']]

    return df_transformed

In [61]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

# ... (your existing functions: log_progress, extract, transform, load_to_csv, load_to_db, run_query) ...


# Assuming these variables are defined somewhere:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'  # Updated URL
table_attribs = ["Country", "GDP_USD_millions"]  # Updated table attributes
csv_path = 'Countries_by_GDP.csv'
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'

# Main ETL Process
log_progress('Preliminaries complete. Initiating ETL process')  # Log: Declaring known values

df = extract(url, table_attribs)  # Call extract()

log_progress('Data extraction complete. Initiating Transformation process')  # Log: Call extract()

df = transform(df)  # Call transform()

log_progress('Data transformation complete. Initiating loading process')  # Log: Call transform()

load_to_csv(df, csv_path)  # Call load_to_csv()

log_progress('Data saved to CSV file')  # Log: Call load_to_csv()

sql_connection = sqlite3.connect(db_name)  # Initiate SQLite3 connection

log_progress('SQL Connection initiated.')  # Log: Initiate SQLite3 connection

load_to_db(df, sql_connection, table_name)  # Call load_to_db()

log_progress('Data loaded to Database as table. Running the query')  # Log: Call load_to_db()

query_statement = f"SELECT * from {table_name} WHERE GDP_USD_billions >= 100"
run_query(query_statement, sql_connection)  # Call run_query()

log_progress('Process Complete.')  # Log: Call run_query()

sql_connection.close()  # Close SQLite3 connection

SELECT * from Countries_by_GDP WHERE GDP_USD_billions >= 100
Empty DataFrame
Columns: [Country, GDP_USD_billions]
Index: []
